# A/B Test Calculator: Design & Analysis

**TL;DR:** This tool answers two questions. Before the test: *how many users and days do I need?* After the test: *can we ship this?* Pick the mode at the top, fill in the fields, press Calculate.

---

## Two modes

- **Design** – before the test. Calculates the required sample size and duration.
- **Analysis** – after the test. Gives a clear ship / don't ship decision with charts.

## Non-inferiority checkbox

By default the calculator runs a classic A/B test (superiority): *is the treatment better or worse than control?*

Turn on **"Include non-inferiority analysis"** when your question is different: *we don't expect this change to improve the metric – we just want to make sure it doesn't hurt more than X*. Typical cases: UI simplification, tech migrations, removing features. This adds one extra field – the margin – and a second chart.

If you are not sure whether you need it, you probably don't. Start with the default mode.

---

## Design Mode

| Field | What it means | Example |
|---|---|---|
| **Baseline CR (%)** | Current conversion rate in the control experience | 11.30 |
| **MDE (p.p.)** | The smallest improvement you want to be able to detect | 0.10 |
| **NI margin (p.p.)** *(only if NI is on)* | Maximum acceptable drop – the worst-case decline you are willing to accept. Agree on this with the team before the test. | 0.30 |
| **NI method** *(only if NI is on)* | **Standard** matches Sealed Envelope when you enter the same alpha there – use this by default. **Advanced** uses separate variances under H0 and H1. Depending on the baseline, it can give a smaller or a larger sample size than Standard. | Standard |
| **Alpha** | False positive rate. 0.05 is standard. Superiority is two-sided; non-inferiority is one-sided (90% CI at 0.05). | 0.05 |
| **Power** | Probability of detecting a real effect. 0.80 is standard. | 0.80 |
| **Ratio (treatment/control)** | Treatment users per one control user. 1.0 = equal split. | 1.0 |
| **Daily traffic** | Total users entering the experiment per day (both groups). Used to estimate duration. | 10,000 |

**Output:** sample size per group, total users, and estimated days. With NI on, you get two columns (superiority and non-inferiority) and a recommendation – always run for the larger number, so one test covers both questions.

---

## Analysis Mode

Enter the actual results after the test is finished. Different group sizes are fine – just set the planned split in **Ratio**.

| Field | What it means | Example |
|---|---|---|
| **Conversions control** | Conversions in the control group | 56,500 |
| **N control** | Total users in the control group | 500,000 |
| **Conversions treatment** | Conversions in the treatment group | 56,250 |
| **N treatment** | Total users in the treatment group | 500,000 |
| **NI margin (p.p.)** *(only if NI is on)* | The same margin you agreed on before the test | 0.30 |
| **Alpha** | The same alpha you used in Design. Must match. | 0.05 |
| **Ratio (treatment/control)** | The planned split, same as in Design. 1.0 = 50/50, 0.25 = 80/20 (control/treatment), 4.0 = 20/80. Used for the SRM check. | 1.0 |

**Output:** chart(s), a decision banner, and a text report.

### Decision – default mode (superiority only)

| # | Result | Decision |
|---|---|---|
| 1 | Treatment better | **Ship** |
| 2 | Not significant | **Inconclusive: keep control** – no difference detected |
| 3 | Treatment worse | **Don't ship** |

### Decision – with non-inferiority on

| # | Superiority result | NI shown? | Decision | What it means |
|---|---|---|---|---|
| 1 | Treatment better | Yes | **Ship** | Treatment is better. |
| 2 | Not significant | Yes | **Safe to ship** | No difference detected, and the CI confirms the drop is within the margin. |
| 3 | Not significant | No | **Don't ship** | Cannot guarantee the drop is acceptable. |
| 4 | Treatment worse | Yes | **Discuss** | Significantly worse, but within the margin. Discuss with the product team. |
| 5 | Treatment worse | No | **Don't ship** | Worse beyond the acceptable limit. |
| 6 | Treatment worse | No | **Don't ship** | Worse, and the CI crosses the margin: a drop beyond the limit cannot be ruled out. |

Non-inferiority is a one-sided test: with alpha = 0.05 it uses the lower bound of the **90% CI** (the superiority chart keeps the two-sided 95% CI).

The calculator highlights which row applies to your test. If the SRM check fails, no row is highlighted and the decision is **SRM FAILURE** (see below).

### Automatic warnings

**SRM check (sample ratio mismatch).** Runs on every analysis. It compares the actual group sizes with the planned split from **Ratio**. If the difference is statistically significant (SRM p-value < 0.001), the decision is replaced with **SRM FAILURE** and no ship / don't ship decision is shown.

SRM is about significance, not size: on large samples even a 0.5% imbalance can be a real bug. It usually means an allocation, logging, or filtering problem. Find the cause first – the results can't be trusted until then.

Two things to check:
- **Ratio is set correctly.** If the split was not 50/50 and Ratio is left at 1.0, the check will fail on every run.
- **Which users are counted.** The check is most reliable on all assigned users. If you analyze only triggered or filtered users, also check the split on all assigned users.

**Borderline check.** If the result is very close to the decision boundary (p-value near alpha, or the CI edge near zero or near the margin), you'll see a yellow "Borderline result" note. It means: the conclusion is formally correct, but a small change in data could flip it. Treat such results with extra care – consider collecting more data before a high-stakes decision.

---

## Two things to remember

**Margin is a product decision, not a statistical one.** It means: "we are OK with losing at most this much in exchange for the benefits of the change." Set it before the test starts, together with the PM. Never adjust it after seeing the results.

**Alpha must be the same in Design and Analysis.** If you designed with alpha = 0.05, analyze with 0.05. Changing it after seeing results invalidates the test.

---

**Assumptions:** binary metric (conversion rate), independent observations, large-sample approximation, fixed-horizon test (no peeking). Calculates one metric at a time – works for both primary and guardrail metrics. Run each guardrail as a separate calculation with its own margin.

---

### Changelog

**NI and labels update.** Non-inferiority now uses one-sided alpha, the industry standard: the decision uses the 90% CI at alpha = 0.05, and the NI sample size uses z(1 - alpha) in both methods. Before, it used the 95% CI and z(1 - alpha/2), which matched Sealed Envelope only at alpha = 2.5%. Same alpha now gives a smaller N (e.g. 137,708 instead of 174,823 per group at 11.3% baseline and 0.30 p.p. margin), and some old "Don't ship" NI results become "Safe to ship". Labels renamed: "No action" -> "Inconclusive: keep control", "Safe to ship*" -> "Discuss". New row 6: significantly worse and the CI crosses the margin (was shown as row 3).

**SRM update.** The SRM check now always runs (before, it ran only when the imbalance was above 1%, so it missed real problems on large samples). Analysis mode has a new **Ratio** field. A failed SRM check now blocks the decision. If you run tests with an unequal split, set Ratio – otherwise the decision will be blocked.

In [ ]:
#@title A/B Test Calculator: Design & Analysis { run: "auto", display-mode: "form" }

import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output


# ══════════════════════════════════════════════════════════════
# CORE STATISTICAL FUNCTIONS
# ══════════════════════════════════════════════════════════════

def _wilson_interval(successes, n, z):
    """Wilson score interval for a single proportion."""
    p_hat = successes / n
    denom = 1 + z ** 2 / n
    center = (p_hat + z ** 2 / (2 * n)) / denom
    hw = (z / denom) * np.sqrt(p_hat * (1 - p_hat) / n + z ** 2 / (4 * n ** 2))
    return center - hw, center + hw


def _newcombe_ci(conv_c, n_c, conv_t, n_t, alpha):
    """
    Newcombe (1998) Method 10 (square-and-add):
    CI for difference (treatment - control).
    """
    z = stats.norm.ppf(1 - alpha / 2)
    l1, u1 = _wilson_interval(conv_c, n_c, z)  # control
    l2, u2 = _wilson_interval(conv_t, n_t, z)  # treatment

    cr_control = conv_c / n_c
    cr_treatment = conv_t / n_t
    diff = cr_treatment - cr_control

    ci_lower = diff - np.sqrt((cr_treatment - l2) ** 2 + (u1 - cr_control) ** 2)
    ci_upper = diff + np.sqrt((u2 - cr_treatment) ** 2 + (cr_control - l1) ** 2)

    return cr_control, cr_treatment, diff, ci_lower, ci_upper


def _pooled_p_value(conv_c, n_c, conv_t, n_t):
    """Two-sided superiority p-value (pooled z-test under H0: cr_c = cr_t)."""
    diff = conv_t / n_t - conv_c / n_c
    p_pooled = (conv_c + conv_t) / (n_c + n_t)
    se = np.sqrt(p_pooled * (1 - p_pooled) * (1 / n_c + 1 / n_t))
    if se == 0:
        return 1.0
    return 2 * (1 - stats.norm.cdf(abs(diff / se)))


def _classify(ci_lower, ni_ci_lower, ni_ci_upper, margin, sig, diff, ni_enabled):
    """
    Classify the result. Returns (row, label, interpretation).
    Superiority: (1 - alpha) CI and the two-sided p-value.
    Non-inferiority: one-sided alpha, i.e. the (1 - 2*alpha) CI (90% at alpha = 0.05).
    """
    if not ni_enabled:
        # Superiority only: three outcomes
        if sig and diff > 0:
            return "1", "SHIP", "Treatment is significantly better than control."
        elif sig and diff < 0:
            return "3", "DON'T SHIP", "Treatment is significantly worse than control."
        else:
            return "2", "INCONCLUSIVE: KEEP CONTROL", (
                "No statistically significant difference detected. "
                "The CI includes zero. Keep control."
            )

    # Superiority + NI: six outcomes
    worse = sig and diff < 0
    ni_passed = ni_ci_lower > -margin
    if ci_lower > 0:
        return "1", "SHIP", "Treatment is significantly better than control."
    elif ni_passed and not worse:
        return "2", "SAFE TO SHIP", (
            "No significant difference. "
            "The CI does not indicate a drop beyond the acceptable margin."
        )
    elif ni_passed and worse:
        return "4", "DISCUSS: WORSE, BUT WITHIN MARGIN", (
            "Treatment is significantly worse, but within the acceptable margin. "
            "Discuss with the product team before shipping."
        )
    elif worse and ni_ci_upper < -margin:
        return "5", "DON'T SHIP", "Treatment is worse than control beyond the acceptable margin."
    elif worse:
        return "6", "DON'T SHIP", (
            "Treatment is significantly worse, and we cannot rule out a drop "
            "larger than the margin."
        )
    else:
        return "3", "DON'T SHIP", (
            "No significant difference, but we cannot rule out a drop larger "
            "than the margin. The CI crosses the non-inferiority boundary."
        )


def _check_srm(n_c, n_t, expected_ratio=1.0, threshold=0.001):
    """
    Sample ratio mismatch check (chi-squared goodness of fit).
    Always runs: SRM is about significance, not the size of the imbalance.
    Returns warning string or None.
    """
    total = n_c + n_t
    expected_c = total / (1 + expected_ratio)
    expected_t = total - expected_c
    _, srm_p = stats.chisquare([n_c, n_t], f_exp=[expected_c, expected_t])
    if srm_p < threshold:
        observed_ratio = n_t / n_c
        return (
            f"SRM FAILURE: ratio T/C = {observed_ratio:.4f} "
            f"(expected {expected_ratio:.4f}), SRM p-value = {srm_p:.2e}. "
            f"Do not interpret results until the cause is found."
        )
    return None


def _check_borderline(p_value, alpha, ci_lower, ci_upper, margin, ni_enabled, ni_ci_lower=None):
    """
    Detect borderline results. Returns list of warning strings.
    Borderline = small changes in data could flip the conclusion.
    """
    warnings = []
    ci_width = ci_upper - ci_lower
    edge_zone = ci_width * 0.10  # 10% of CI width

    # p-value near alpha: [alpha*0.7, alpha*1.3]
    if alpha * 0.7 <= p_value <= alpha * 1.3:
        warnings.append(
            f"p-value ({p_value:.4f}) is close to alpha ({alpha}). "
            f"A small change in data could flip significance."
        )

    # CI edge near zero
    dist_lower_to_zero = abs(ci_lower)
    dist_upper_to_zero = abs(ci_upper)
    if min(dist_lower_to_zero, dist_upper_to_zero) < edge_zone:
        warnings.append(
            f"CI edge is close to zero. "
            f"The conclusion about direction could change with more data."
        )

    # CI edge near margin (NI mode only)
    if ni_enabled and margin > 0:
        ni_low = ci_lower if ni_ci_lower is None else ni_ci_lower
        dist_lower_to_margin = abs(ni_low - (-margin))
        if dist_lower_to_margin < edge_zone:
            warnings.append(
                f"CI lower bound is close to -margin. "
                f"The non-inferiority conclusion could change with more data."
            )

    return warnings


# ══════════════════════════════════════════════════════════════
# DESIGN MODE
# ══════════════════════════════════════════════════════════════

def run_design(baseline_cr, mde, margin, alpha, power, ratio, daily_traffic, ni_enabled, ni_method="standard"):
    clear_output(wait=True)
    display(ui)

    errors = []
    if not (0 < baseline_cr < 1):
        errors.append(f"Baseline CR must be between 0% and 100%. Got {baseline_cr:.4%}.")
    if mde <= 0:
        errors.append(f"MDE must be positive. Got {mde}.")
    if ni_enabled and margin <= 0:
        errors.append(f"Margin must be positive. Got {margin}.")
    if not (0 < alpha < 1):
        errors.append(f"Alpha must be between 0 and 1. Got {alpha}.")
    if ni_enabled and 0.5 <= alpha < 1:
        errors.append(f"With non-inferiority on, alpha must be below 0.5 (NI uses the (1 - 2*alpha) CI). Got {alpha}.")
    if not (0 < power < 1):
        errors.append(f"Power must be between 0 and 1. Got {power}.")
    if ratio <= 0:
        errors.append(f"Ratio must be positive. Got {ratio}.")
    if 0 < baseline_cr < 1 and mde > 0 and baseline_cr + mde >= 1:
        errors.append(f"Baseline + MDE = {baseline_cr + mde:.4f} >= 1.")
    if ni_enabled and 0 < baseline_cr < 1 and margin > 0 and baseline_cr - margin <= 0:
        errors.append(f"Baseline - Margin = {baseline_cr - margin:.4f} <= 0.")
    if errors:
        for e in errors:
            print(f"  ERROR: {e}")
        return

    z_a = stats.norm.ppf(1 - alpha / 2)
    z_b = stats.norm.ppf(power)
    cr_control = baseline_cr
    cr_treatment = baseline_cr + mde

    # Superiority sample size
    p_bar = (cr_control + cr_treatment) / 2
    se_h0_sup = np.sqrt(p_bar * (1 - p_bar) * (1 + 1 / ratio))
    se_h1_sup = np.sqrt(cr_control * (1 - cr_control) + cr_treatment * (1 - cr_treatment) / ratio)
    n_c_sup = int(np.ceil(((z_a * se_h0_sup + z_b * se_h1_sup) / mde) ** 2))
    n_t_sup = int(np.ceil(n_c_sup * ratio))
    n_total_sup = n_c_sup + n_t_sup

    ci_pct = f"{(1 - alpha) * 100:.0f}%"
    days_sup = int(np.ceil(n_total_sup / daily_traffic)) if daily_traffic > 0 else None

    print("=" * 64)
    print("              EXPERIMENT DESIGN")
    print("=" * 64)
    print(f"  Baseline CR:      {baseline_cr:.4%}")
    print(f"  MDE:              {mde * 100:.4f} p.p.")
    if ni_enabled:
        print(f"  NI margin:        {margin * 100:.4f} p.p.")
        method_label = (
            "Standard (Blackwelder, matches Sealed Envelope at the same alpha)"
            if ni_method == "standard"
            else "Advanced (separate H0/H1 variances)"
        )
        print(f"  NI method:        {method_label}")
    print(f"  Alpha:            {alpha}  ({ci_pct} two-sided CI)")
    if ni_enabled:
        print(f"  NI alpha:         {alpha} one-sided  ({(1 - 2 * alpha) * 100:.0f}% CI)")
    print(f"  Power:            {power:.0%}")
    print(f"  Group ratio:      {ratio} (treatment / control)")
    if daily_traffic > 0:
        print(f"  Daily traffic:    {daily_traffic:,}")
    print("-" * 64)

    if ni_enabled:
        # NI sample size: one-sided alpha (same as Sealed Envelope with the same alpha)
        z_a_ni = stats.norm.ppf(1 - alpha)
        if ni_method == "standard":
            # Blackwelder: same variance for both hypotheses
            # (matches Sealed Envelope and most online calculators)
            cr_t_expected = cr_control  # assume no true difference
            var_ni = cr_control * (1 - cr_control) + cr_t_expected * (1 - cr_t_expected) / ratio
            n_c_ni = int(np.ceil(((z_a_ni + z_b) ** 2 * var_ni) / margin ** 2))
        else:
            # Separate variances under H0 and H1.
            # Can be smaller or larger than Standard, depending on baseline.
            cr_t_h0 = cr_control - margin
            cr_t_h1 = cr_control
            se_h0_ni = np.sqrt(cr_control * (1 - cr_control) + cr_t_h0 * (1 - cr_t_h0) / ratio)
            se_h1_ni = np.sqrt(cr_control * (1 - cr_control) + cr_t_h1 * (1 - cr_t_h1) / ratio)
            n_c_ni = int(np.ceil(((z_a_ni * se_h0_ni + z_b * se_h1_ni) / margin) ** 2))
        n_t_ni = int(np.ceil(n_c_ni * ratio))
        n_total_ni = n_c_ni + n_t_ni
        days_ni = int(np.ceil(n_total_ni / daily_traffic)) if daily_traffic > 0 else None
        n_max = max(n_total_sup, n_total_ni)

        print()
        print(f"  {'':30s} {'Superiority':>15s}   {'Non-Inferiority':>15s}")
        print(f"  {'─' * 30} {'─' * 15}   {'─' * 15}")
        print(f"  {'N control':30s} {n_c_sup:>15,}   {n_c_ni:>15,}")
        print(f"  {'N treatment':30s} {n_t_sup:>15,}   {n_t_ni:>15,}")
        print(f"  {'N total':30s} {n_total_sup:>15,}   {n_total_ni:>15,}")
        if daily_traffic > 0:
            print(f"  {'Estimated duration':30s} {days_sup:>12,} d   {days_ni:>12,} d")
        print()
        print("  Recommendation:")
        if daily_traffic > 0:
            days_max = int(np.ceil(n_max / daily_traffic))
            print(f"  Run for {days_max:,} days ({n_max:,} users) to cover both tests.")
        else:
            print(f"  Collect {n_max:,} users total to cover both tests.")
    else:
        print()
        print(f"  {'N control':30s} {n_c_sup:>15,}")
        print(f"  {'N treatment':30s} {n_t_sup:>15,}")
        print(f"  {'N total':30s} {n_total_sup:>15,}")
        if daily_traffic > 0:
            print(f"  {'Estimated duration':30s} {days_sup:>12,} d")
        print()
        print("  Recommendation:")
        if daily_traffic > 0:
            print(f"  Run for {days_sup:,} days ({n_total_sup:,} users).")
        else:
            print(f"  Collect {n_total_sup:,} users total.")

    print("=" * 64)


# ══════════════════════════════════════════════════════════════
# ANALYSIS MODE
# ══════════════════════════════════════════════════════════════

def run_analysis(conv_c, conv_t, n_c, n_t, margin, alpha, ni_enabled, expected_ratio=1.0):
    clear_output(wait=True)
    display(ui)

    errors = []
    if n_c <= 0:
        errors.append(f"N control must be positive. Got {n_c}.")
    if n_t <= 0:
        errors.append(f"N treatment must be positive. Got {n_t}.")
    if conv_c < 0:
        errors.append(f"Conversions control must be >= 0. Got {conv_c}.")
    if conv_t < 0:
        errors.append(f"Conversions treatment must be >= 0. Got {conv_t}.")
    if n_c > 0 and conv_c > n_c:
        errors.append(f"Conversions control ({conv_c}) > N control ({n_c}).")
    if n_t > 0 and conv_t > n_t:
        errors.append(f"Conversions treatment ({conv_t}) > N treatment ({n_t}).")
    if ni_enabled and margin <= 0:
        errors.append(f"Margin must be positive. Got {margin}.")
    if not (0 < alpha < 1):
        errors.append(f"Alpha must be between 0 and 1. Got {alpha}.")
    if ni_enabled and 0.5 <= alpha < 1:
        errors.append(f"With non-inferiority on, alpha must be below 0.5 (NI uses the (1 - 2*alpha) CI). Got {alpha}.")
    if expected_ratio <= 0:
        errors.append(f"Ratio must be positive. Got {expected_ratio}.")
    if errors:
        for e in errors:
            print(f"  ERROR: {e}")
        return

    # Compute
    cr_c, cr_t, diff, ci_lower, ci_upper = _newcombe_ci(conv_c, n_c, conv_t, n_t, alpha)
    # NI is one-sided at alpha: it uses the (1 - 2*alpha) CI (90% at alpha = 0.05)
    _, _, _, ni_ci_lower, ni_ci_upper = _newcombe_ci(conv_c, n_c, conv_t, n_t, 2 * alpha)
    sup_p_value = _pooled_p_value(conv_c, n_c, conv_t, n_t)
    sig = sup_p_value < alpha
    row_num, decision, interpretation = _classify(
        ci_lower, ni_ci_lower, ni_ci_upper, margin if ni_enabled else 0, sig, diff, ni_enabled
    )

    ci_pct = f"{(1 - alpha) * 100:.0f}%"
    ni_ci_pct = f"{(1 - 2 * alpha) * 100:.0f}%"

    # SRM check: blocks the decision
    srm_warning = _check_srm(n_c, n_t, expected_ratio)
    if srm_warning:
        decision = "SRM FAILURE"
        row_num = None
        interpretation = (
            "Sample ratio mismatch. Results are not interpretable. "
            "Find the cause before any decision."
        )
        print(f"\n  \u26a0 {srm_warning}\n")

    # Borderline check
    borderline_warnings = _check_borderline(
        sup_p_value, alpha, ci_lower, ci_upper,
        margin if ni_enabled else 0, ni_enabled, ni_ci_lower=ni_ci_lower
    )

    # ── Colors ──
    GREEN = "#2e8b57"
    RED = "#d84a3a"
    NEUTRAL = "#9aa0a6"
    AMBER = "#d4a017"

    # ── Layout ──
    if ni_enabled:
        fig = plt.figure(figsize=(10, 6.8))
        gs = fig.add_gridspec(3, 1, height_ratios=[0.32, 1, 1], hspace=0.28)
    else:
        fig = plt.figure(figsize=(10, 4.2))
        gs = fig.add_gridspec(2, 1, height_ratios=[0.4, 1], hspace=0.28)

    # ── Decision banner ──
    ax_banner = fig.add_subplot(gs[0])
    ax_banner.set_xlim(0, 1)
    ax_banner.set_ylim(0, 1)
    banner_text_color = (
        RED if decision in ("DON'T SHIP", "SRM FAILURE")
        else NEUTRAL if decision.startswith("INCONCLUSIVE")
        else AMBER if decision.startswith("DISCUSS")
        else "#444"
    )
    ax_banner.text(0.5, 0.55, f"Decision: {decision}",
                   ha="center", va="center", fontsize=13, color=banner_text_color,
                   bbox=dict(boxstyle="round,pad=0.25", facecolor="white",
                            edgecolor="#cccccc", linewidth=1))
    if srm_warning:
        ax_banner.text(0.5, 0.15, "\u26a0 SRM – check allocation before reading anything below",
                       ha="center", va="center", fontsize=9, color=RED)
    elif borderline_warnings:
        ax_banner.text(0.5, 0.15, "\u26a0 Borderline result – see details below",
                       ha="center", va="center", fontsize=9, color=AMBER)
    ax_banner.axis("off")

    # ── Shared axis limits ──
    ci_l, ci_u = ci_lower * 100, ci_upper * 100
    ni_l, ni_u = ni_ci_lower * 100, ni_ci_upper * 100
    d = diff * 100
    pad = max(0.04, (ci_u - ci_l) * 0.15)
    if ni_enabled:
        m = -margin * 100
        xlim_l = min(ci_l, m) - pad
    else:
        xlim_l = ci_l - pad
    xlim_r = max(ci_u, 0) + pad

    # ── Helper: draw one chart ──
    def draw_ci(ax, bar_color, show_margin, title, info_text, lo=None, hi=None):
        lo = ci_l if lo is None else lo
        hi = ci_u if hi is None else hi
        ax.plot([lo, hi], [0, 0], color=bar_color, linewidth=2.2, solid_capstyle="butt")
        tick_h = 0.07
        ax.plot([lo, lo], [-tick_h, tick_h], color=bar_color, linewidth=1.8)
        ax.plot([hi, hi], [-tick_h, tick_h], color=bar_color, linewidth=1.8)
        ax.plot(d, 0, "o", color=bar_color, markersize=7, zorder=5)
        ax.axvline(0, color="#b0b0b0", linestyle=":", linewidth=1)

        if show_margin:
            ax.axvline(m, color="#9a9a9a", linestyle="--", linewidth=1.2)
            ax.text(m, 0.16, "-margin", ha="center", va="bottom",
                    fontsize=7.5, color="#9a9a9a")

        ax.text(0.98, 0.92, info_text, transform=ax.transAxes, ha="right", va="top",
                fontsize=9, fontfamily="monospace", color=bar_color,
                bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                         edgecolor="#dddddd", linewidth=0.8))

        ax.set_xlim(xlim_l, xlim_r)
        ax.set_ylim(-0.25, 0.25)
        ax.set_yticks([])
        ax.set_title(title, fontsize=12, loc="left", color="#333", pad=10)
        for s in ["top", "right", "left"]:
            ax.spines[s].set_visible(False)

    # ── Superiority chart ──
    ax1 = fig.add_subplot(gs[1])
    sup_color = GREEN if (sig and diff > 0) else RED if (sig and diff < 0) else NEUTRAL
    sup_info = f"p = {sup_p_value:.3f} ({'sig' if sig else 'ns'})"
    draw_ci(ax1, sup_color, False, f"Superiority ({ci_pct} CI)", sup_info)

    if ni_enabled:
        ax1.tick_params(axis="x", which="both", bottom=False, labelbottom=False)
        # ── NI chart ──
        ni_passed = ni_ci_lower > -margin
        ax2 = fig.add_subplot(gs[2])
        ni_color = GREEN if ni_passed else RED
        ni_info = f"NI: {'yes' if ni_passed else 'no'}"
        draw_ci(ax2, ni_color, True,
                f"Non-Inferiority ({ni_ci_pct} CI, one-sided \u03b1 = {alpha})",
                ni_info, ni_l, ni_u)
        ax2.set_xlabel("Treatment \u2212 Control, p.p.", fontsize=10, color="#555", labelpad=8)
    else:
        ax1.set_xlabel("Treatment \u2212 Control, p.p.", fontsize=10, color="#555", labelpad=8)

    fig.subplots_adjust(top=0.95, bottom=0.08)
    plt.show()

    # ── Borderline warnings ──
    if borderline_warnings:
        print()
        for bw in borderline_warnings:
            print(f"  \u26a0 BORDERLINE: {bw}")
        print()

    # ── Text report ──
    print("=" * 72)
    print(f"  Control:   {conv_c:,} / {n_c:,} = {cr_c:.4%}")
    print(f"  Treatment: {conv_t:,} / {n_t:,} = {cr_t:.4%}")
    print(f"  Diff (T-C): {diff * 100:+.4f} p.p.")
    print(f"  {ci_pct} CI:    [{ci_lower * 100:.4f}, {ci_upper * 100:.4f}] p.p.  (Newcombe)")
    print(f"  Superiority p-value: {sup_p_value:.4f} (alpha={alpha}, two-sided)")
    if ni_enabled:
        worst_case_drop = max(0.0, -ni_ci_lower)
        print(f"  NI margin: {margin * 100:.4f} p.p.")
        print(f"  {ni_ci_pct} CI:    [{ni_ci_lower * 100:.4f}, {ni_ci_upper * 100:.4f}] p.p.  (Newcombe, for NI)")
        print(f"  NI decision is based on the lower bound of the {ni_ci_pct} CI (one-sided alpha = {alpha})")
        print(f"  Worst-case drop from CI lower bound: {worst_case_drop * 100:.4f} p.p.")
    print("-" * 72)

    if ni_enabled:
        matrix = [
            ("1", "Treatment better", "Yes", "Ship",            "Treatment is better."),
            ("2", "Not significant",  "Yes", "Safe to ship",    "CI above -margin."),
            ("3", "Not significant",  "No",  "Don't ship",      "CI crosses -margin."),
            ("4", "Treatment worse",  "Yes", "Discuss",         "Worse, but within margin."),
            ("5", "Treatment worse",  "No",  "Don't ship",      "Beyond margin."),
            ("6", "Treatment worse",  "No",  "Don't ship",      "CI crosses -margin."),
        ]

        print(f"\n  {'#':<4}{'Superiority':<20}{'NI?':<8}{'Decision':<16}{'Comment'}")
        print(f"  {'─' * 4}{'─' * 20}{'─' * 8}{'─' * 16}{'─' * 24}")
        for r, sup, ni, dec, comment in matrix:
            marker = " >>>" if r == row_num else "    "
            print(f"{marker}{r:<4}{sup:<20}{ni:<8}{dec:<16}{comment}")
    else:
        matrix = [
            ("1", "Treatment better",     "Ship",            "Treatment is better."),
            ("2", "Not significant",       "Inconclusive",    "No difference detected. Keep control."),
            ("3", "Treatment worse",       "Don't ship",      "Treatment is worse."),
        ]

        print(f"\n  {'#':<4}{'Superiority':<24}{'Decision':<16}{'Comment'}")
        print(f"  {'─' * 4}{'─' * 24}{'─' * 16}{'─' * 24}")
        for r, sup, dec, comment in matrix:
            marker = " >>>" if r == row_num else "    "
            print(f"{marker}{r:<4}{sup:<24}{dec:<16}{comment}")

    print(f"\n  >>> {decision}: {interpretation}")
    print("=" * 72)


# ══════════════════════════════════════════════════════════════
# WIDGETS
# ══════════════════════════════════════════════════════════════

style = {"description_width": "170px"}
layout = widgets.Layout(width="460px")

w_mode = widgets.ToggleButtons(
    options=["Design", "Analysis"],
    value="Design",
    button_style="",
    layout=widgets.Layout(width="460px"),
)

# NI toggle (shared between modes)
w_ni = widgets.Checkbox(
    value=False,
    description="Include non-inferiority analysis",
    indent=False,
    layout=widgets.Layout(width="460px"),
)

# NI sample size method (Design mode only)
w_ni_method = widgets.Dropdown(
    options=[
        ("Standard – matches Sealed Envelope (Blackwelder)", "standard"),
        ("Advanced – separate H0/H1 variances", "advanced"),
    ],
    value="standard",
    description="NI method:",
    style=style,
    layout=layout,
)

# Design
d_baseline = widgets.FloatText(value=11.30, description="Baseline CR (%):",          style=style, layout=layout)
d_mde      = widgets.FloatText(value=0.10,  description="MDE (p.p.):",               style=style, layout=layout)
d_margin   = widgets.FloatText(value=0.30,  description="NI margin (p.p.):",         style=style, layout=layout)
d_alpha    = widgets.FloatText(value=0.05,  description="Alpha:",                    style=style, layout=layout)
d_power    = widgets.FloatText(value=0.80,  description="Power:",                    style=style, layout=layout)
d_ratio    = widgets.FloatText(value=1.0,   description="Ratio (treatment/control):", style=style, layout=layout)
d_traffic  = widgets.IntText(value=10000,   description="Daily traffic:",             style=style, layout=layout)

# Analysis
a_conv_c = widgets.IntText(value=56500,   description="Conversions control:",   style=style, layout=layout)
a_n_c    = widgets.IntText(value=500000,  description="N control:",             style=style, layout=layout)
a_conv_t = widgets.IntText(value=56250,   description="Conversions treatment:", style=style, layout=layout)
a_n_t    = widgets.IntText(value=500000,  description="N treatment:",           style=style, layout=layout)
a_margin = widgets.FloatText(value=0.30,  description="NI margin (p.p.):",      style=style, layout=layout)
a_alpha  = widgets.FloatText(value=0.05,  description="Alpha:",                 style=style, layout=layout)
a_ratio  = widgets.FloatText(value=1.0,   description="Ratio (treatment/control):", style=style, layout=layout)

# Panel
panel = widgets.Output()

def update_panel(*args):
    panel.clear_output()
    with panel:
        ni = w_ni.value
        if w_mode.value == "Design":
            if ni:
                display(widgets.VBox([d_baseline, d_mde, d_margin, w_ni_method, d_alpha, d_power, d_ratio, d_traffic]))
            else:
                display(widgets.VBox([d_baseline, d_mde, d_alpha, d_power, d_ratio, d_traffic]))
        else:
            if ni:
                display(widgets.VBox([a_conv_c, a_n_c, a_conv_t, a_n_t, a_margin, a_alpha, a_ratio]))
            else:
                display(widgets.VBox([a_conv_c, a_n_c, a_conv_t, a_n_t, a_alpha, a_ratio]))

w_mode.observe(update_panel, names="value")
w_ni.observe(update_panel, names="value")
update_panel()

# Button
w_btn = widgets.Button(description="Calculate", button_style="primary",
                       layout=widgets.Layout(width="460px", height="38px"))

def on_click(_):
    if w_mode.value == "Design":
        run_design(
            baseline_cr=d_baseline.value / 100,
            mde=d_mde.value / 100,
            margin=d_margin.value / 100 if w_ni.value else 0,
            alpha=d_alpha.value,
            power=d_power.value,
            ratio=d_ratio.value,
            daily_traffic=d_traffic.value,
            ni_enabled=w_ni.value,
            ni_method=w_ni_method.value,
        )
    else:
        run_analysis(
            conv_c=a_conv_c.value,
            conv_t=a_conv_t.value,
            n_c=a_n_c.value,
            n_t=a_n_t.value,
            margin=a_margin.value / 100 if w_ni.value else 0,
            alpha=a_alpha.value,
            ni_enabled=w_ni.value,
            expected_ratio=a_ratio.value,
        )

w_btn.on_click(on_click)

# Assemble
ui = widgets.VBox([
    widgets.HTML(
    "<h3 style='margin-bottom:2px'>A/B Test Calculator</h3>"
    "<p style='color:#666; margin-top:0; margin-bottom:2px; font-size:13px'>"
    "Superiority + Non-Inferiority &nbsp;|&nbsp; Newcombe CI &nbsp;|&nbsp; Binary conversion</p>"
    "<p style='color:#999; margin-top:0; margin-bottom:2px; font-size:11px'>"
    "Assumptions: binary metric, independent observations, large-sample approximation, "
    "fixed-horizon test. Calculates one metric at a time – works for both primary and "
    "guardrail metrics. Run each guardrail separately with its own margin. "
    "Non-inferiority uses one-sided alpha (90% CI at alpha = 0.05), same as Sealed Envelope.</p>"
    "<p style='color:#999; margin-top:0; margin-bottom:8px; font-size:11px'>"
    "SRM check runs on every analysis. If the split is not 50/50, set Ratio (treatment/control), "
    "e.g. 0.25 for 80/20. Use counts of assigned users, not only triggered or filtered ones. "
    "If SRM fails, the decision is blocked.</p>"
),
    w_mode,
    w_ni,
    widgets.HTML("<div style='height:4px'></div>"),
    panel,
    widgets.HTML("<div style='height:4px'></div>"),
    w_btn,
])

display(ui)